In [1]:
from pathlib import Path
DATA_DIR = Path.cwd() / "Data"
print(list(DATA_DIR.glob("*.csv")))

[PosixPath('/Users/tsinww/Desktop/123/intern/tech/recsys/H&M/pythonProject1/Data/customers.csv'), PosixPath('/Users/tsinww/Desktop/123/intern/tech/recsys/H&M/pythonProject1/Data/articles.csv'), PosixPath('/Users/tsinww/Desktop/123/intern/tech/recsys/H&M/pythonProject1/Data/transactions_train.csv'), PosixPath('/Users/tsinww/Desktop/123/intern/tech/recsys/H&M/pythonProject1/Data/sample_submission.csv')]


In [3]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch version:", torch.__version__)

PyTorch version: 2.15.0


In [5]:
# from IPython.display import display
# 
# pd.set_option("display.max_columns", None)
# 
# files = [
#     "articles.csv",
#     "customers.csv",
#     "transactions_train.csv",
#     "sample_submission.csv",
# ]
# 
# for filename in files:
#     print(f"\n—— {filename} ——")
# 
#     preview = pd.read_csv(
#         DATA_DIR / filename,
#         nrows=5,
#         dtype={"article_id": "string", "customer_id": "string"}
#     )
# 
#     display(preview)

In [ ]:
# Read only the date column, in chunks, to keep memory usage low
n_rows = 0
first_date = None
last_date = None

for chunk in pd.read_csv(
    DATA_DIR / "transactions_train.csv",
    usecols=["t_dat"],
    chunksize=500_000
):
    n_rows += len(chunk)
    chunk_min = chunk["t_dat"].min()
    chunk_max = chunk["t_dat"].max()

    first_date = (
        chunk_min if first_date is None else min(first_date, chunk_min)
    )
    last_date = (
        chunk_max if last_date is None else max(last_date, chunk_max)
    )

print(f"Purchase records: {n_rows:,}")
print(f"First date: {first_date}")
print(f"Last date:  {last_date}")

parts = []

for chunk in pd.read_csv(
    DATA_DIR / "transactions_train.csv",
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
    chunksize=500_000
):
    recent = chunk.loc[chunk["t_dat"] >= "2020-08-26"]
    if not recent.empty:
        parts.append(recent)

transactions = pd.concat(parts, ignore_index=True)
del parts

print(f"Loaded {len(transactions):,} purchase records")
display(transactions.head())

history = transactions.loc[
    transactions["t_dat"] < "2020-09-16"
].copy()

validation = transactions.loc[
    transactions["t_dat"] >= "2020-09-16"
].copy()

print(f"Earlier purchases: {len(history):,}")
print(f"Held-out purchases: {len(validation):,}")

In [ ]:
# Read only the date column, in chunks, to keep memory usage low
n_rows = 0
first_date = None
last_date = None

for chunk in pd.read_csv(
    DATA_DIR / "transactions_train.csv",
    usecols=["t_dat"],
    chunksize=500_000
):
    n_rows += len(chunk)
    chunk_min = chunk["t_dat"].min()
    chunk_max = chunk["t_dat"].max()

    first_date = (
        chunk_min if first_date is None else min(first_date, chunk_min)
    )
    last_date = (
        chunk_max if last_date is None else max(last_date, chunk_max)
    )

print(f"Purchase records: {n_rows:,}")
print(f"First date: {first_date}")
print(f"Last date:  {last_date}")

parts = []

for chunk in pd.read_csv(
    DATA_DIR / "transactions_train.csv",
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
    chunksize=500_000
):
    recent = chunk.loc[chunk["t_dat"] >= "2020-08-26"]
    if not recent.empty:
        parts.append(recent)

transactions = pd.concat(parts, ignore_index=True)
del parts

print(f"Loaded {len(transactions):,} purchase records")
display(transactions.head())

history = transactions.loc[
    transactions["t_dat"] < "2020-09-16"
].copy()

validation = transactions.loc[
    transactions["t_dat"] >= "2020-09-16"
].copy()

print(f"Earlier purchases: {len(history):,}")
print(f"Held-out purchases: {len(validation):,}")

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

rng = np.random.default_rng(42)
torch.manual_seed(42)

# One positive example per distinct customer–article pair
pairs = history[["customer_id", "article_id"]].drop_duplicates()
#drop duplicates pairs but customer and article can show up repeatedly. see the factorization later.

customer_idx, customer_ids = pd.factorize(pairs["customer_id"])
#ids is to numerical label distinct customers
#idx is to record the numerical labels along the name list, so it can contain duplicated numerics
article_idx, article_ids = pd.factorize(pairs["article_id"])

n_customers = len(customer_ids)
n_articles = len(article_ids)

print(f"Customers: {n_customers:,}")
print(f"Articles: {n_articles:,}")
print(f"Distinct purchased pairs: {len(pairs):,}")

In [ ]:
# Encode a pair as a single integer so we can check membership
#n_customer * n_articles pairs in total, ofc, this excludes the customer and articles that never show up.
#the seq number(key) of each pair can be encoded as customer_id * n_articles + article_id; 
#and if we only want the keys of the purchased pairs: we just need to call the existing customer_idx and article_idx as below.
purchased_keys = np.unique(
    customer_idx.astype(np.int64) * n_articles + article_idx
)
#np.unique once again removes the duplicates, we already did the removal of duplicated pairs so this step is unnecessary.

neg_per_pur = 3
negative_customers = np.repeat(customer_idx, neg_per_pur)
#generate a sequence of length 3*n_purchased_pairs, i.e. 3 negatives per purchased pair
#not 3 negatives per customer! Why? suppose Alice bought 1 items(1 positive sample) and Bob bought 100 items(100 positive sample), 
#Alice will have 3 negative sample and Bob will have 300 negative samples
#if it is 3 negative per customer then Alice and Bob will both have 3 negative samples but Bob have way more positive samples.
#This will balance the positive and negative samples within one customer (not to balance the weight between customers, so Alice is not necessary for illustration here oops)
#having too little/too imbalanced negative/positive ratio will possibly bring irrelevant videos to Bob's recommendation!
negative_articles = rng.integers(
    0, n_articles, size=len(negative_customers)
)#generate a seq of the same length but put random article keys on it, will replace/resample those who are actually positive later.

# Resample any negatives that are actually observed purchases
#basically we sample the negatives uniformly, not using the hard negative sampling
while True:
    negative_keys = (
        negative_customers.astype(np.int64) * n_articles
        + negative_articles
    )
    invalid = np.isin(negative_keys, purchased_keys)#return entry that is in pur_keys with True, not in then False

    if not invalid.any():#if invalid does not have any True.
        break

    negative_articles[invalid] = rng.integers(#boolean indexing. Only operates on position that is true.
        0, n_articles, size=invalid.sum()
    )

train_customers = np.concatenate([customer_idx, negative_customers])
train_articles = np.concatenate([article_idx, negative_articles])

labels = np.concatenate([
    np.ones(len(customer_idx), dtype=np.float32),
    np.zeros(len(negative_customers), dtype=np.float32)#put zeros are negative samples; we use binary cross entropy loss
])

dataset = TensorDataset(
    torch.tensor(train_customers, dtype=torch.long),
    torch.tensor(train_articles, dtype=torch.long),
    torch.tensor(labels)
)
#whenever using pytorch, we need to convert the data input into tensor.

loader = DataLoader(dataset, batch_size=8192, shuffle=True)

print(f"Training examples, including negatives: {len(dataset):,}")

In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_customers, n_articles, dim=32):
        super().__init__()

        # Create the two trainable embedding tables, fill it with random numbers
        self.customer_embeddings = nn.Embedding(n_customers, dim)
        self.article_embeddings = nn.Embedding(n_articles, dim)

        # Initialize their entries with small random numbers, overwrites the numbers with new random valuew from N(0,0.1^2).
        nn.init.normal_(self.customer_embeddings.weight, mean=0, std=0.1)
        nn.init.normal_(self.article_embeddings.weight, mean=0, std=0.1)
        #why? why initialize with normal distribution?

    def forward(self, customers, articles):
        # Look up vectors for the customer–article pairs
        u = self.customer_embeddings(customers) #=self.customer_embeddings.weight[customers]
        v = self.article_embeddings(articles) #=self.article_embeddings.weight[customers]

        # One dot-product score per pair
        scores = (u * v).sum(dim=1)
        #need to check, why???? why u and v??? only diagonals?
        return scores

model = MatrixFactorization(
    n_customers=n_customers,
    n_articles=n_articles,
    dim=32
)

#print(model)

# Use a GPU if available; otherwise use the CPU; The first line selects a CUDA GPU if available; otherwise, it selects the CPU. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#moves the model’s parameters to that device.
model = model.to(device)
# define the loss function: binary cross entropy L = -y log p - (1-y)log(1-p); 
#\hat{y} would be the label. \hat{p} would be sigmoid of learned scores 
loss_fn = nn.BCEWithLogitsLoss()

# create an Adam optimizer to optimize the parameters later.
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
#lr learning rate \eta: the step x_{t} = x_{t-1} - \eta \Delta L

print("Running on:", device)

n_epochs = 3

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0

    for customers, articles, targets in loader:
        # Move this batch to the same device as the model
        customers = customers.to(device)
        articles = articles.to(device)
        targets = targets.to(device)

        # Clear gradients left over from the previous batch
        optimizer.zero_grad()

        # Call forward: retrieve embeddings and compute dot products
        scores = model(customers, articles)

        # Compare scores with the purchase labels
        loss = loss_fn(scores, targets)

        # Compute gradients for the embedding parameters
        loss.backward()

        # Update their values
        optimizer.step()

        total_loss += loss.item() * len(targets)

    average_loss = total_loss / len(loader.dataset)
    print(f"Epoch {epoch + 1}: loss = {average_loss:.4f}")
    
# define the loss function: binary cross entropy L = -y log p - (1-y)log(1-p); 
#\hat{y} would be the label. \hat{p} would be sigmoid of learned scores 
loss_fn = nn.BCEWithLogitsLoss()

# create an Adam optimizer to optimize the parameters later.
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
#lr learning rate \eta: the step x_{t} = x_{t-1} - \eta \Delta L

print("Running on:", device)

n_epochs = 3

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0

    for customers, articles, targets in loader:
        # Move this batch to the same device as the model
        customers = customers.to(device)
        articles = articles.to(device)
        targets = targets.to(device)

        # Clear gradients left over from the previous batch
        optimizer.zero_grad()

        # Call forward: retrieve embeddings and compute dot products
        scores = model(customers, articles)

        # Compare scores with the purchase labels
        loss = loss_fn(scores, targets)

        # Compute gradients for the embedding parameters
        loss.backward()

        # Update their values
        optimizer.step()

        total_loss += loss.item() * len(targets)

    average_loss = total_loss / len(loader.dataset)
    print(f"Epoch {epoch + 1}: loss = {average_loss:.4f}")